In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field, TypeAdapter, ValidationError
from typing import Literal
import json

In [2]:
load_dotenv()

True

In [3]:
client = OpenAI()

In [4]:
def calculate_gst(amount: float, gst_rate: float) -> float:
    return amount * gst_rate / 100

In [5]:
def calculate_discount(amount: float, discount_rate: float) -> float:
    return amount * discount_rate / 100

In [6]:
tools = [
    {
        "type": "function",
        "name": "calculate_gst",
        "description": "Calculate GST amount from a price and GST percentage.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {
                    "type": "number",
                    "description": "Price before GST"
                },
                "gst_rate": {
                    "type": "number",
                    "description": "GST percentage"
                }
            },
            "required": ["amount", "gst_rate"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "calculate_discount",
        "description": "Calculate discount amount from a price and discount percentage.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {
                    "type": "number",
                    "description": "Price before discount"
                },
                "discount_rate": {
                    "type": "number",
                    "description": "dicount percentage"
                }
            },
            "required": ["amount", "discount_rate"],
            "additionalProperties": False
        },
        "strict": True
    }
]

In [7]:
user_question = input("Ask GPT...")

In [8]:
user_question

"'Calculate a 10% GST on ₹3,850.'"

In [9]:
response = client.responses.create(
    model="gpt-6-luna",
    input=user_question,
    tools=tools
)

In [10]:
response.output[0]

ResponseFunctionToolCall(arguments='{"amount":3850,"gst_rate":10}', call_id='call_Bnwa41uYQuKOd12V6vpVzqim', name='calculate_gst', type='function_call', id='fc_0248b9697311fdbd006ac9da180ed487d09399c1e3393ec686', async_=None, caller=None, namespace=None, status='completed')

In [11]:
tool_registry = {
    "calculate_gst": calculate_gst,
    "calculate_discount": calculate_discount
}

In [12]:
tool_call = next(
    item for item in response.output
    if item.type == "function_call"
)

In [13]:
tool_call

ResponseFunctionToolCall(arguments='{"amount":3850,"gst_rate":10}', call_id='call_Bnwa41uYQuKOd12V6vpVzqim', name='calculate_gst', type='function_call', id='fc_0248b9697311fdbd006ac9da180ed487d09399c1e3393ec686', async_=None, caller=None, namespace=None, status='completed')

In [14]:
class GstArguments(BaseModel):
    amount: float = Field(..., ge=0, description="The base amount for GST calculation")
    gst_rate: float = Field(..., ge=0, le=100, description="The GST percentage rate")

class DiscountArguments(BaseModel):
    amount: float = Field(..., ge=0, description="The base amount for discount calculation")
    discount_rate: float = Field(..., ge=0, le=100,description="The discount percentage rate")

In [15]:
schema_registry = {
    "calculate_gst": GstArguments,
    "calculate_discount": DiscountArguments
}

In [16]:
class ToolNotFound(Exception):
    pass

In [17]:
tool_outputs = []
execution_failed = False

for tool_call in response.output:
    if tool_call.type != "function_call":
        continue
    
    try:
        if tool_call.name not in tool_registry:
            print(f'Unknown tool: {tool_call.name}')
            raise ToolNotFound

        schema = schema_registry[tool_call.name]
        args = schema.model_validate_json(tool_call.arguments)
        function = tool_registry[tool_call.name]
        result = function(**args.model_dump())
        print(f"Tool Name: {tool_call.name}")
        print(f"Arguments: {args}")
        print(f"Result: {result}")
        tool_outputs.append(
            {
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": str(result)
            }
        )

    except ValidationError as e:
        print(f"Validation failed: {e}")
        execution_failed = True
    except ToolNotFound as e:
        print(f"Unknown Tool Used: {e}")
        execution_failed = True

Tool Name: calculate_gst
Arguments: amount=3850.0 gst_rate=10.0
Result: 385.0


In [18]:
tool_outputs

[{'type': 'function_call_output',
  'call_id': 'call_Bnwa41uYQuKOd12V6vpVzqim',
  'output': '385.0'}]

In [19]:
if execution_failed:
    final_response = "Tool execution failed. Final LLM call skipped."
else:
    final_response = client.responses.create(
        model="gpt-6-luna",
        previous_response_id=response.id,
        input=tool_outputs,
        tools=tools
    )

In [20]:
if execution_failed:
    print(final_response)
else:
    print(final_response.output_text)

10% GST on ₹3,850 is **₹385**. The total including GST is **₹4,235**.


### Validation

In [21]:
try:
    result1 = GstArguments.model_validate({
        "amount": 45000,
        "gst_rate": 18
    })
    print(result1)
except ValidationError as e:
    print(f'Error: {e}')

try:
    result2 = DiscountArguments.model_validate({
    "amount": 2000,
    "discount_rate": 150
    })
    print(result2)
except ValidationError as e:
    print(f'Error: {e}')

try:
    result3 = DiscountArguments.model_validate({
    "amount": -500,
    "discount_rate": 10
    })
    print(result3)
except ValidationError as e:
    print(f'Error: {e}')


amount=45000.0 gst_rate=18.0
Error: 1 validation error for DiscountArguments
discount_rate
  Input should be less than or equal to 100 [type=less_than_equal, input_value=150, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal
Error: 1 validation error for DiscountArguments
amount
  Input should be greater than or equal to 0 [type=greater_than_equal, input_value=-500, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal
